# AHN pre-processing: Splitting large LAZ files

- Workshop: **High performance computing with Python and RS-DAT, SURF Utrecht**

- Date: March 19, 2026 

In this notebook, we demonstrate the use of dask and low-level dask abstractions to support the pre-processing of massive data sets whose constituent files can easily expand beyond memory. Using the dask bag abstraction that supports user defined python objects we create a distrubted lazy retiling of [AHN data](https://www.ahn.nl/) for subsequent processing.

**NOTE**: The workflow illustrated in this notebook is used in production, but is run on much larger collections of laz files than in this example.

In [ ]:
import laspy
import numpy as np
import pathlib
import os

import dask.bag as db

## Set input parameters

For subsequent processing it is beneficial if individual data files fit in memory, as not all tools in the lidar processing chain support lazy loading. We therefore define a maximum desired file size of 250 MB (in compressed laz format) which roughly correspoinds to a maximum decompressed value of 2.5 GB (depending on the compression factor).

In [ ]:
# desired max file size (in bytes)
max_filesize = 250 * 2**20 # 250 MB

In [ ]:
# input path (original laz files)
input_path = pathlib.Path("/project/remotesensing/Share/hpcdat_ws1/ahn_tiles/laz")

# output path (split into smaller files), please set the correct folder name using your account ID: <stu[ID]>, e.g. stu01
output_path = pathlib.Path("/project/remotesensing/Share/hpcdat_ws1/users/<stu[ID]>")

In [ ]:
laz_files = list(input_path.glob("*.LAZ"))
print(f"Splitting {len(laz_files)} LAZ files")

## Setup Cluster

Setup Dask cluster used for splitting.

In [ ]:
#Drag cluster object here

In [ ]:
#Alternative: specify connection to existing cluster
#from dask.distributed import Client

#client = Client("tcp://10.0.0.52:33961")
#client

## Splitting

AHN 3 is the third complete airborne lidar scanning survey of the Netherlands, attaining lidar point densities of ~20 pts/m^2 over the entire country. This equates to ~5 TB of compressed data.

AHN 3 is made available as (compressed) laz files on the TOP10NL grid, i.e 6.25km by 5km tiles, with sizes reaching > 6GB compressed. With a compresssion factor of ~7, tiles can expand to >42 GB in RAM.

To facilitate further processing for purposes such as macro-ecology, which uses the lidar data to characterize habitat structure on scales of 10m, it is desirable to split the data to smaller units, as not all tools 

In [ ]:
# make sure output dir exists
output_path.mkdir(exist_ok=True)

In [ ]:
# approximate compression factor
LAZ_COMPRESSION_FACTOR = 7

def split_strategy(filepath, max_filesize, out_dir="./"):
    """
    Set up splitting strategy for a LAS/LAZ file.

    By only parsing the metadata of each LAZ file, we figure out which points
    should end up in which file. 
    """
    with laspy.open(filepath) as f:
        bytes_per_point = (
            f.header.point_format.num_standard_bytes +
            f.header.point_format.num_extra_bytes
        )
        n_points = f.header.point_count
    n_points_target = int(
        max_filesize * LAZ_COMPRESSION_FACTOR / bytes_per_point
    )
    filename = os.path.basename(filepath)
    stem, ext = os.path.splitext(filename)
    return [
        (filepath, f"{out_dir}/{stem}-{n:02d}{ext}", offset, n_points_target)
        for n, offset in enumerate(range(0, n_points, n_points_target))
    ]
    
def save_chunk_to_laz_file(in_filename,
                           out_filename,
                           offset,
                           n_points):
    """Read points from a LAS/LAZ file and write them to a new file."""
    points = np.array([])
    with laspy.open(in_filename) as in_file:
        with laspy.open(out_filename,
                        mode="w",
                        header=in_file.header) as out_file:
            in_file.seek(offset)
            points = in_file.read_points(n_points)
            out_file.write_points(points)
    return len(points)

In [ ]:
# set up calculation
files = db.from_sequence(laz_files)
input_args = files \
    .map(
        split_strategy,
        max_filesize=max_filesize,
        out_dir=output_path.as_posix(),
    ) \
    .flatten() \
    .unzip(4)  # unpack input arguments 
res = db.map(save_chunk_to_laz_file, *input_args)

In [ ]:
# run the splitting
tot_points = res.compute()

In [ ]:
# splitted points
sum(tot_points)